# CARC data and run status

This notebook does not train a model. It checks that the frozen data, CARC reference cache, and run folders are ready. Training is submitted from the terminal with `scripts/submit_carc.py`.

## 1. Find the repository and artifact directory

Start Jupyter from the repository after setting `MFR_OUTPUT_DIR` to the same absolute path used by the Slurm submission commands.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd

REPO = next((p for p in [Path.cwd(), Path.cwd().parent]
             if (p/'configs/experiment_protocol.json').exists()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the mfr-dpo repository')
if 'MFR_OUTPUT_DIR' not in os.environ:
    raise EnvironmentError('Set MFR_OUTPUT_DIR before starting Jupyter')
ARTIFACT_DIR = Path(os.environ['MFR_OUTPUT_DIR']).expanduser().resolve()
sys.path.insert(0, str(REPO/'src'))
protocol = json.loads((REPO/'configs/experiment_protocol.json').read_text())
print('Repository:', REPO)
print('Artifacts:', ARTIFACT_DIR)
print('Protocol:', protocol['protocol_version'])

## 2. Verify the frozen data

Success means every JSONL file still matches the committed manifest and every split passes the schema and leakage checks.

In [ ]:
import mfr_data
splits = mfr_data.load_splits(REPO/protocol['data_dir'])
pd.DataFrame([{'Dataset': dataset, **{part.title(): len(frame) for part, frame in parts.items()}}
              for dataset, parts in splits.items()])

## 3. Check the CARC reference cache

The cache is built once on the same L40S GPU type used for training. Each experiment also performs a live canary check before training starts.

In [ ]:
cache = ARTIFACT_DIR/'cache'/f"reference_{protocol['data_version']}.csv"
manifest = cache.with_suffix('.manifest.json')
print('Cache:', cache, 'FOUND' if cache.exists() else 'MISSING')
print('Manifest:', manifest, 'FOUND' if manifest.exists() else 'MISSING')
if manifest.exists():
    display(pd.Series(json.loads(manifest.read_text()), name='value').to_frame())

## 4. List completed and partial runs

A run is final only when it contains `COMPLETE.json`. Resubmitting the same order and seed skips completed methods and resumes at the first unfinished stage.

In [ ]:
rows=[]
for folder in sorted((ARTIFACT_DIR/'runs').glob('*')):
    settings=folder/'settings.json'; results=folder/'results.csv'
    saved=json.loads(settings.read_text()) if settings.exists() else {}
    maximum=pd.read_csv(results).stage.max() if results.exists() else None
    rows.append({'Run':folder.name,'Order':saved.get('order_id'),'Seed':saved.get('seed'),
                 'Method':saved.get('method'),'Last finished stage':maximum,
                 'Complete':(folder/'COMPLETE.json').exists()})
pd.DataFrame(rows) if rows else pd.DataFrame(columns=['Run','Order','Seed','Method','Last finished stage','Complete'])